In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

In [ ]:
# ==========================================
# 1. CONFIGURATION
# ==========================================
DATA_DIR = './processed_data' # Ensure this matches where you ran make_dataset.py
TRAIN_FILE = os.path.join(DATA_DIR, 'quickdraw_train.npz')
TEST_FILE = os.path.join(DATA_DIR, 'quickdraw_test.npz')

BATCH_SIZE = 128
EPOCHS = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {DEVICE}")


In [ ]:
# ==========================================
# 2. DATASET CLASS (The NPZ Loader)
# ==========================================

class QuickDrawDataset(Dataset):
    def __init__(self, file_path, mode='train'):
        """
        Args:
            file_path (str): Path to the .npz file
            mode (str): 'train' (loads images & labels) or 'test' (loads images only)
        """
        self.mode = mode

        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Could not find file: {file_path}")

        print(f"Loading {mode} data from {file_path}...")
        data = np.load(file_path)

        if mode == 'train':
            # Load x_train and y_train
            self.x = data['x_train']
            self.y = data['y_train']
            self.classes = data['class_names']
            print(f"Loaded {len(self.x)} training samples. Classes: {len(self.classes)}")

        elif mode == 'test':
            # Load test_images (for leaderboard inference)
            self.x = data['test_images']
            self.y = None
            print(f"Loaded {len(self.x)} test images.")

        # Pre-processing:
        # Convert to Float Tensor and Normalize (0-255 -> 0-1)
        self.x = torch.from_numpy(self.x).float() / 255.0

        if self.y is not None:
            self.y = torch.from_numpy(self.y).long()

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        img = self.x[idx]
        if self.mode == 'train':
            label = self.y[idx]
            return img, label
        else:
            return img

In [ ]:
# ==========================================
# 3. PREPARE DATALOADERS
# ==========================================

CLASSES = ['apple', 'baseballbat', 'basketball', 'clock', 'compass', 'cookie', 'donut', 'ladder', 'mountain', 'pizza', 'rabbit', 'soccerball', 'spider', 't-shirt', 'wheel']

# 1. Load the Training Data
full_train_dataset = QuickDrawDataset(TRAIN_FILE, mode='train')
NUM_CLASSES = len(full_train_dataset.classes)

# 2. Create Validation Split (80% Train / 20% Val)
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

# 3. Create Loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples: {len(train_dataset)} | Validation samples: {len(val_dataset)}")

In [ ]:
# ==========================================
# 4. YOUR IMPLEMENTATION HERE
# ==========================================

import csv
import json
import random
import time
from pathlib import Path

OUTPUT_DIR = Path("notebook_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)


def get_activation(name):
    key = name.lower()
    if key == 'relu':
        return nn.ReLU()
    if key == 'gelu':
        return nn.GELU()
    if key in {'leakyrelu', 'leaky_relu'}:
        return nn.LeakyReLU(negative_slope=0.1)
    raise ValueError(f"Unsupported activation: {name}")


class FlexibleMLP(nn.Module):
    def __init__(self, input_size, num_classes, hidden_dims, activation='relu', dropout=0.0, batch_norm=False):
        super().__init__()
        layers = []
        prev_dim = input_size

        for dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, dim))
            if batch_norm:
                layers.append(nn.BatchNorm1d(dim))
            layers.append(get_activation(activation))
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev_dim = dim

        layers.append(nn.Linear(prev_dim, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.net(x)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def run_epoch(model, loader, criterion, optimizer=None):
    train_mode = optimizer is not None
    model.train(train_mode)

    total_loss, total_correct, total_samples = 0.0, 0, 0

    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)

        if train_mode:
            optimizer.zero_grad(set_to_none=True)

        logits = model(X)
        loss = criterion(logits, y)

        if train_mode:
            loss.backward()
            optimizer.step()

        batch_size = y.size(0)
        total_loss += float(loss.item()) * batch_size
        total_correct += int((logits.argmax(dim=1) == y).sum().item())
        total_samples += batch_size

    return total_loss / total_samples, total_correct / total_samples


def train_experiment(config):
    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False)

    model = FlexibleMLP(
        input_size=784,
        num_classes=NUM_CLASSES,
        hidden_dims=config['hidden_dims'],
        activation=config['activation'],
        dropout=config['dropout'],
        batch_norm=config['batch_norm'],
    ).to(DEVICE)

    params = count_parameters(model)
    print(f"\n[{config['name']}] Parameter count: {params:,}")
    if params > 3_000_000:
        raise ValueError(f"Model {config['name']} violates 3M parameter cap with {params:,} params")

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['epochs'])

    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [],
        'lr': [], 'epoch_time_sec': []
    }

    best_val_acc = -1.0
    best_epoch = -1
    best_state = None
    no_improve = 0

    for epoch in range(1, config['epochs'] + 1):
        start = time.time()

        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer=None)

        scheduler.step()
        epoch_time = time.time() - start

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['lr'].append(optimizer.param_groups[0]['lr'])
        history['epoch_time_sec'].append(epoch_time)

        print(
            f"[{config['name']}] Epoch {epoch:02d}/{config['epochs']} "
            f"| train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
            f"| val_loss={val_loss:.4f} val_acc={val_acc:.4f} "
            f"| {epoch_time:.1f}s"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            no_improve = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1

        if no_improve >= config['patience']:
            print(f"[{config['name']}] Early stopping at epoch {epoch}")
            break

    model.load_state_dict(best_state)

    ckpt_path = OUTPUT_DIR / f"{config['name']}_best.pth"
    torch.save({
        'config': config,
        'model_state_dict': model.state_dict(),
        'best_epoch': best_epoch,
        'best_val_acc': best_val_acc,
        'params': params,
    }, ckpt_path)

    result = {
        'name': config['name'],
        'config': config,
        'model': model,
        'params': params,
        'best_epoch': best_epoch,
        'best_val_acc': best_val_acc,
        'train_acc_at_best': history['train_acc'][best_epoch - 1],
        'val_loss_at_best': history['val_loss'][best_epoch - 1],
        'used_epochs': len(history['train_loss']),
        'total_time_sec': float(sum(history['epoch_time_sec'])),
        'checkpoint': str(ckpt_path),
        'history': history,
    }

    with open(OUTPUT_DIR / f"{config['name']}_history.json", 'w') as f:
        json.dump({
            'name': result['name'],
            'params': result['params'],
            'best_epoch': result['best_epoch'],
            'best_val_acc': result['best_val_acc'],
            'train_acc_at_best': result['train_acc_at_best'],
            'val_loss_at_best': result['val_loss_at_best'],
            'used_epochs': result['used_epochs'],
            'total_time_sec': result['total_time_sec'],
            'checkpoint': result['checkpoint'],
            'history': result['history'],
        }, f, indent=2)

    return result


def plot_history(result):
    history = result['history']
    epochs = np.arange(1, len(history['train_loss']) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, history['train_loss'], label='Train')
    axes[0].plot(epochs, history['val_loss'], label='Validation')
    axes[0].set_title(f"{result['name']} Loss")
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Cross-Entropy')
    axes[0].legend()

    axes[1].plot(epochs, history['train_acc'], label='Train')
    axes[1].plot(epochs, history['val_acc'], label='Validation')
    axes[1].set_title(f"{result['name']} Accuracy")
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{result['name']}_curves.png", dpi=180)
    plt.show()


def plot_comparison(results):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for result in results:
        epochs = np.arange(1, len(result['history']['val_acc']) + 1)
        axes[0].plot(epochs, result['history']['val_acc'], label=result['name'])
        axes[1].plot(epochs, result['history']['val_loss'], label=result['name'])

    axes[0].set_title('Validation Accuracy Comparison')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()

    axes[1].set_title('Validation Loss Comparison')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Cross-Entropy')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "model_comparison.png", dpi=180)
    plt.show()


# -----------------------------
# Experiment definitions
# -----------------------------
experiments = [
    {
        'name': 'pancake',
        'hidden_dims': [2048],
        'activation': 'relu',
        'dropout': 0.25,
        'batch_norm': False,
        'lr': 1e-3,
        'weight_decay': 1e-4,
        'epochs': 24,
        'batch_size': 256,
        'patience': 7,
    },
    {
        'name': 'tower',
        'hidden_dims': [256, 256, 256, 256, 256, 256],
        'activation': 'leaky_relu',
        'dropout': 0.10,
        'batch_norm': True,
        'lr': 1.2e-3,
        'weight_decay': 5e-4,
        'epochs': 30,
        'batch_size': 256,
        'patience': 8,
    },
    {
        'name': 'champion',
        'hidden_dims': [1024, 768, 512, 256],
        'activation': 'gelu',
        'dropout': 0.25,
        'batch_norm': True,
        'lr': 8e-4,
        'weight_decay': 1e-3,
        'epochs': 34,
        'batch_size': 256,
        'patience': 9,
    },
]


# -----------------------------
# Train all models
# -----------------------------
all_results = []
for cfg in experiments:
    print("\n" + "=" * 90)
    print(f"Training {cfg['name']} with hidden_dims={cfg['hidden_dims']}")
    result = train_experiment(cfg)
    all_results.append(result)
    plot_history(result)

plot_comparison(all_results)


# -----------------------------
# Select champion model
# -----------------------------
champion_result = max(all_results, key=lambda x: x['best_val_acc'])
model = champion_result['model']  # This variable is used by the inference cell below.
print("\nSelected champion:")
print(f"  Name: {champion_result['name']}")
print(f"  Params: {champion_result['params']:,}")
print(f"  Best epoch: {champion_result['best_epoch']}")
print(f"  Best val acc: {champion_result['best_val_acc']:.4f}")
print(f"  Train acc @ best: {champion_result['train_acc_at_best']:.4f}")


# -----------------------------
# Confusion matrix and top confused pairs
# -----------------------------
def collect_predictions(model, loader):
    model.eval()
    y_true, y_pred, x_all = [], [], []
    with torch.no_grad():
        for X, y in loader:
            logits = model(X.to(DEVICE))
            preds = logits.argmax(dim=1).cpu()
            y_true.append(y.cpu())
            y_pred.append(preds)
            x_all.append(X.cpu())
    return torch.cat(x_all), torch.cat(y_true), torch.cat(y_pred)


def confusion_matrix_np(y_true, y_pred, num_classes):
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        cm[int(t), int(p)] += 1
    return cm


val_eval_loader = DataLoader(val_dataset, batch_size=512, shuffle=False)
x_val_tensor, y_true_tensor, y_pred_tensor = collect_predictions(model, val_eval_loader)

cm = confusion_matrix_np(y_true_tensor.numpy(), y_pred_tensor.numpy(), NUM_CLASSES)
np.save(OUTPUT_DIR / "champion_confusion_matrix.npy", cm)

plt.figure(figsize=(10, 8))
plt.imshow(cm, cmap='Blues')
plt.title('Champion Validation Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks(np.arange(NUM_CLASSES), CLASSES, rotation=45, ha='right')
plt.yticks(np.arange(NUM_CLASSES), CLASSES)
plt.colorbar()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "champion_confusion_matrix.png", dpi=200)
plt.show()

pair_scores = []
for i in range(NUM_CLASSES):
    for j in range(i + 1, NUM_CLASSES):
        pair_scores.append((cm[i, j] + cm[j, i], i, j))

pair_scores.sort(reverse=True, key=lambda x: x[0])
top2_pairs = pair_scores[:2]

print("\nTop 2 confused class pairs:")
confusion_report = []
for rank, (score, i, j) in enumerate(top2_pairs, start=1):
    item = {
        'rank': rank,
        'class_a': CLASSES[i],
        'class_b': CLASSES[j],
        'combined_confusions': int(score),
        'a_to_b': int(cm[i, j]),
        'b_to_a': int(cm[j, i]),
    }
    confusion_report.append(item)
    print(
        f"  {rank}. {CLASSES[i]} vs {CLASSES[j]} -> total={score}, "
        f"{CLASSES[i]}->{CLASSES[j]}={cm[i, j]}, {CLASSES[j]}->{CLASSES[i]}={cm[j, i]}"
    )

with open(OUTPUT_DIR / 'top2_confusions.json', 'w') as f:
    json.dump(confusion_report, f, indent=2)


# Save a few misclassified samples for each top confused pair.
x_val_np = x_val_tensor.numpy()
y_true_np = y_true_tensor.numpy()
y_pred_np = y_pred_tensor.numpy()

for pair_idx, (_, i, j) in enumerate(top2_pairs, start=1):
    mask = ((y_true_np == i) & (y_pred_np == j)) | ((y_true_np == j) & (y_pred_np == i))
    selected = np.where(mask)[0][:12]
    if len(selected) == 0:
        continue

    rows, cols = 2, 6
    fig, axes = plt.subplots(rows, cols, figsize=(14, 5))
    axes = axes.flatten()

    for k, sample_idx in enumerate(selected):
        axes[k].imshow(x_val_np[sample_idx].reshape(28, 28), cmap='gray')
        t = CLASSES[y_true_np[sample_idx]]
        p = CLASSES[y_pred_np[sample_idx]]
        axes[k].set_title(f"T:{t}\nP:{p}", fontsize=8)
        axes[k].axis('off')

    for k in range(len(selected), len(axes)):
        axes[k].axis('off')

    plt.suptitle(f"Confused Pair {pair_idx}: {CLASSES[i]} vs {CLASSES[j]}")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"confused_pair_{pair_idx}_{CLASSES[i]}_vs_{CLASSES[j]}.png", dpi=180)
    plt.show()


# -----------------------------
# Summary table for report
# -----------------------------
summary_rows = []
for r in all_results:
    summary_rows.append({
        'model': r['name'],
        'params': r['params'],
        'used_epochs': r['used_epochs'],
        'best_epoch': r['best_epoch'],
        'best_val_acc': r['best_val_acc'],
        'train_acc_at_best': r['train_acc_at_best'],
        'val_loss_at_best': r['val_loss_at_best'],
        'total_time_sec': r['total_time_sec'],
    })

summary_csv = OUTPUT_DIR / 'model_summary.csv'
with open(summary_csv, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(summary_rows[0].keys()))
    writer.writeheader()
    writer.writerows(summary_rows)

print("\nModel comparison table:")
for row in summary_rows:
    print(
        f"  {row['model']:<9} | params={row['params']:,} | used_epochs={row['used_epochs']:>2} "
        f"| best_val_acc={row['best_val_acc']:.4f} | train_acc@best={row['train_acc_at_best']:.4f}"
    )

report_notes = {
    'width_vs_depth': (
        "Even though a single hidden layer can approximate continuous functions in theory, "
        "deep models are often more parameter-efficient for compositional image features and usually generalize better."
    ),
    'overfitting_strategy': (
        "Applied dropout, weight decay, early stopping, and batch normalization to reduce overfitting."
    ),
    'top_confusions': confusion_report,
    'selected_champion': {
        'name': champion_result['name'],
        'params': champion_result['params'],
        'best_epoch': champion_result['best_epoch'],
        'best_val_acc': champion_result['best_val_acc'],
    }
}
with open(OUTPUT_DIR / 'report_notes.json', 'w') as f:
    json.dump(report_notes, f, indent=2)

print(f"\nSaved artifacts under: {OUTPUT_DIR.resolve()}")



In [ ]:
# ==========================================
# 5. INFERENCE & LEADERBOARD VERIFICATION
# ==========================================
print("\n" + "="*40)
print("   GENERATING SUBMISSION FILE")
print("="*40)

# 1. Load Test Images
test_dataset = QuickDrawDataset(TEST_FILE, mode='test')
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


def get_predictions(model, loader):
    model.eval()
    model.to(DEVICE)
    preds = []
    with torch.no_grad():
        for batch in loader:
            X = batch.to(DEVICE)
            outputs = model(X)
            predicted = outputs.argmax(dim=1)
            preds.extend(predicted.cpu().numpy().tolist())
    return preds


# 2. Run Inference
print("Running inference on test set...")
predictions = get_predictions(model, test_loader)

# 3. Save as Comma-Separated Text File
submission_file = "submission.txt"
print(f"Saving predictions to '{submission_file}'...")

submission_string = ",".join(map(str, predictions))
with open(submission_file, "w") as f:
    f.write(submission_string)

# Also save a copy in notebook_outputs
with open(OUTPUT_DIR / "submission.txt", "w") as f:
    f.write(submission_string)

print(f"Total predictions: {len(predictions)}")
print("-> Copy & paste the results from 'submission.txt' to the leaderboard portal.")



In [ ]:
def print_model_size(model):
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nModel Statistics:")
    print(f"  Total Parameters: {total_params:,}")
    if total_params > 3000000:
        print("  ⚠️ WARNING: You are over the 3M parameter limit!")
    else:
        print("  ✅ Parameter count is within limits.")

print_model_size(model)